# `InteractionMeshRetargeter` 当前求解问题说明

本文对应 [`src/interaction_mesh_retargeter.py`](./src/interaction_mesh_retargeter.py) 中的 `InteractionMeshRetargeter`，目标是给出当前代码实际构造并求解的优化问题。

本文把问题分成四层：

1. 输入数据和 interaction mesh 目标的构造；
2. 固定帧、固定内层迭代点时的局部凸子问题；
3. 目标项、约束项和四元数更新的代码含义；
4. 它与 `Retarget/` 当前 solver 家族之间的关系。

核心：`InteractionMeshRetargeter` 每一帧并不直接解一个全局非线性轨迹优化，而是在当前帧内反复解一个带线性化约束和二次目标的局部凸问题。决策变量是 active configuration slice 的增量 `dqa` 以及辅助 Laplacian 变量 `lap_var`；求解器为 `cvxpy + Clarabel`；求得增量后直接加到 MuJoCo `qpos` 上，并对根四元数归一化。


## 1. 代码入口与调用链

主要入口脚本：

- 单条序列 retarget：[`examples/robot_retarget.py`](./examples/robot_retarget.py)
- 批量 retarget：[`examples/parallel_robot_retarget.py`](./examples/parallel_robot_retarget.py)
- 已保存结果播放：[`viser_player.py`](./viser_player.py)
- 结果评估：[`evaluation/eval_retargeting.py`](./evaluation/eval_retargeting.py)

`examples/robot_retarget.py` 默认导入：

```python
from holosoma_retargeting.src.interaction_mesh_retargeter import InteractionMeshRetargeter
```

因此这一入口使用的是本文说明的原始 `src/interaction_mesh_retargeter.py`，不是 `*_etasp.py`、`*_first_order.py` 或 `*_laplacian_smooth.py` 变体。

主调用链为：

```text
examples/robot_retarget.py
  -> InteractionMeshRetargeter.retarget_motion(...)
       -> for each frame t:
            build source interaction mesh
            compute source Laplacian target
            choose nominal weight and previous-frame warm start
            iterate(...)
              -> solve_single_iteration(...)
                   build local convex problem in cvxpy
                   solve with Clarabel
                   update qpos by additive increment
                   normalize root quaternion
```

`retarget_motion(...)` 是整条动作的外层逐帧循环。`iterate(...)` 是当前帧内部的 sequential convexification / SQP-style 循环。`solve_single_iteration(...)` 是真正构造一个局部凸子问题的函数。


## 2. 符号、数组布局与索引

### 2.1 核心符号总表

| 符号 / 代码名 | 维度 | 含义 |
|---|---:|---|
| $t$ | 标量索引 | 动作序列的离散帧索引，不是连续时间变量，也不是优化变量 |
| $n$ | 标量索引 | 固定帧 $t$ 内的 sequential convexification / SQP-style 局部迭代索引 |
| $q$、`robot_data.qpos` | $n_q$ | MuJoCo generalized position / configuration 向量 |
| $v$、`robot_data.qvel` | $n_v$ | MuJoCo generalized velocity 向量。含 free-joint 三维角速度，因此通常 $n_v\ne n_q$ |
| $\dot q$ | $n_q$ | `qpos` 坐标导数。free-joint 四元数块是 4 维坐标导数，不是 MuJoCo `qvel` |
| $T(q)$ | $n_v\times n_q$ | 当前代码手写的局部映射，满足 $v=T(q)\dot q$ |
| $\mathcal A$、`self.q_a_indices` | $n_a$ 个索引 | active `qpos` slice，只覆盖当前局部问题允许改变的机器人坐标列 |
| $q_a=S_aq$ | $n_a$ | 完整 `qpos` 的 active slice |
| $\Delta q_{a,t}$、`dqa` | $n_a$ | 当前帧局部优化变量；active `qpos` 坐标的有限小增量，不是 `qvel`，也不是乘了时间步长的速度 |
| $J_{p,v}$、`Jp` | $3\times n_v$ | MuJoCo `mj_jac` 返回的点平移速度对 `qvel` 的 Jacobian |
| $J_{p,\dot q}=J_{p,v}T(q)$ | $3\times n_q$ | 点位置对 `qpos` 坐标导数的一阶 Jacobian |
| $J_{p,a}=J_{p,\dot q}[:,\mathcal A]$ | $3\times n_a$ | 实际和 `dqa` 相乘的 active-slice 点位置 Jacobian |

当前 Holosoma 局部子问题没有把 $\Delta q_{a,t}$ 解释为 $v\Delta t$。它直接把 $\Delta q_{a,t}$ 当作 `qpos` 坐标里的小位移，并使用一阶近似 $\delta p\approx J_{p,a}\Delta q_{a,t}$。

### 2.2 时间和迭代索引

- $t$：动作序列的帧索引。
- $n$：固定帧 $t$ 内部的局部迭代索引。
- $q_t^{(n)}$：第 $t$ 帧、第 $n$ 次内层迭代时的完整 MuJoCo configuration。
- $q_{t-1}$：上一帧最终接受的 configuration，对应代码传入 `solve_single_iteration(...)` 的 `q_t_last`。
- $q_t^{locked}$：当前帧被锁定的完整 configuration 模板，对应 `q_locked`。其中 object pose 已经设置为当前帧 augmented object pose，非 active slice 的分量也来自该模板。

### 2.3 完整 configuration

代码使用 MuJoCo `qpos` 布局。默认 free root 采用：

$$
q = \begin{bmatrix}
p_x & p_y & p_z & r_w & r_x & r_y & r_z & q_{j,1} & \cdots & q_{j,m} & \cdots
\end{bmatrix}^\top.
$$

其中：

- $p=(p_x,p_y,p_z)$ 是机器人根平移；
- $r=(r_w,r_x,r_y,r_z)$ 是机器人根四元数，标量在前；
- $q_j$ 是机器人内部关节位置；
- 如果 MuJoCo model 中包含动态物体，物体 free-joint pose 位于 `qpos` 尾部 7 维。

本文把完整 configuration 记为 $q\in\mathbb{R}^{n_q}$。

### 2.4 active slice

`InteractionMeshRetargeter.__init__` 定义：

```python
self.q_a_indices = np.arange(7 + self.q_a_init_idx, 7 + self.task_constants.ROBOT_DOF)
self.nq_a = len(self.q_a_indices)
```

默认 `q_a_init_idx=-7`，因此：

$$
\mathcal{A}=\{0,1,\dots,6+\mathrm{ROBOT\_DOF}\},
$$

active slice 包含机器人根平移、根四元数和全部机器人关节，不包含动态物体尾部 7 维。本文记：

$$
q_{a,t}^{(n)} = q_t^{(n)}[\mathcal{A}] \in \mathbb{R}^{n_a}.
$$

局部优化变量为：

$$
\Delta q_{a,t} \in \mathbb{R}^{n_a},
$$

代码变量名为 `dqa`。

默认配置下，`dqa` 的前 7 维是根 free joint 的 position-coordinate 增量：

$$
\Delta q_{a,t}[0:7]
=
\begin{bmatrix}
\Delta p_x & \Delta p_y & \Delta p_z & \Delta r_w & \Delta r_x & \Delta r_y & \Delta r_z
\end{bmatrix}^\top.
$$

这不是 generalized velocity，也不是三维角速度；四元数部分直接是 4 维 `qpos` 坐标增量。

### 2.5 本文使用的选择矩阵

对任意完整向量 $x\in\mathbb{R}^{n_q}$，定义 active-slice 选择矩阵：

$$
S_a x = x[\mathcal{A}].
$$

代码里通常不显式构造 $S_a$，而是用 NumPy 切片：

```python
x[self.q_a_indices]
```

对 nominal tracking 还有一个更小的选择算子。代码中：

```python
idx = np.array(self.track_nominal_indices, dtype=int)
z = dqa[idx] - (q_a_nominal[idx] - q_a_n_last[idx])
```

本文记这个选择算子为 $\Pi$。它不是代码中的显式矩阵，而是由 `idx` 切片实现：

$$
\Pi x = x[\texttt{track\_nominal\_indices}].
$$

默认 `RobotConfig` 中：

$$
\Pi =
\begin{cases}
\text{选择 }0,1,\dots,18, & \text{robot type 为 g1},\\
\text{选择 }0,1,\dots,6\text{ 和 }11,\dots,22, & \text{robot type 为 t1},\\
\text{空选择}, & \text{其他 robot type 且未显式配置}.
\end{cases}
$$


## 3. 当前代码真正求解的局部问题

### 3.1 当前帧的 source interaction mesh

每一帧 $t$，代码先取人体关键点和物体点构成 source 顶点集合。

设人体关键点为：

$$
H_t = \{h_{t,1},\dots,h_{t,N_r}\}, \qquad h_{t,i}\in\mathbb{R}^3.
$$

设 demo object local points 为：

$$
O^{demo}=\{o^{demo}_1,\dots,o^{demo}_{N_o}\}, \qquad o^{demo}_j\in\mathbb{R}^3.
$$

当 `self.object_name != "ground"` 时，人体关键点会先从 world frame 转到 demo object frame：

$$
\bar h_{t,i} = R_{O,t}^{demo\top}(h_{t,i}-p_{O,t}^{demo}).
$$

当 `self.object_name == "ground"` 时，代码直接使用 world-frame 人体关键点：

$$
\bar h_{t,i}=h_{t,i}.
$$

source 顶点矩阵为：

$$
V_t^{src}=
\begin{bmatrix}
\bar h_{t,1}^\top\\
\vdots\\
\bar h_{t,N_r}^\top\\
(o_1^{demo})^\top\\
\vdots\\
(o_{N_o}^{demo})^\top
\end{bmatrix}
\in\mathbb{R}^{N\times 3},
\qquad N=N_r+N_o.
$$

代码对 $V_t^{src}$ 做 Delaunay tetrahedralization，并从 tetrahedra 得到 adjacency list。记该邻接结构为 $\mathcal{E}_t$。

对顶点 $i$ 的邻居集合记为 $\mathcal{N}_t(i)$。当前工具函数默认使用 uniform weights，因此 Laplacian 坐标为：

$$
\ell_{t,i}^{src}
=
V_{t,i}^{src}
-
\frac{1}{|\mathcal{N}_t(i)|}
\sum_{j\in\mathcal{N}_t(i)}V_{t,j}^{src}
\in\mathbb{R}^3.
$$

把所有顶点堆叠为：

$$
\hat \ell_t = \operatorname{vec}(L_t V_t^{src})\in\mathbb{R}^{3N},
$$

其中 $L_t\in\mathbb{R}^{N\times N}$ 是由 adjacency list 和 uniform weights 构造的 Laplacian matrix。代码变量对应关系为：

- `source_vertices`：$V_t^{src}$；
- `source_tetrahedra`：Delaunay tetrahedra；
- `adj_list`：$\mathcal{E}_t$；
- `target_laplacian`：$L_tV_t^{src}$；
- `target_lap_vec`：$\hat\ell_t$。

### 3.2 当前迭代点处的 robot-side interaction mesh

在固定当前迭代点 $q_t^{(n)}$ 后，代码构造 robot-side 顶点集合。

设 retargeting 映射中的机器人 link keypoint 位置为：

$$
p_i(q_t^{(n)})\in\mathbb{R}^3,
\qquad i=1,\dots,N_r.
$$

当 `obj_frame=True` 时，`_calc_manipulator_jacobians(...)` 把这些机器人点表达在 augmented object frame 中；当 object 是 ground 时，点保持在 world frame。为了统一记号，本文把该表达坐标系称为 $X_t$，即：

$$
p_i^{X_t}(q_t^{(n)})
=
\begin{cases}
R_{O,t}^{aug\top}\bigl(p_i^W(q_t^{(n)})-p_{O,t}^{aug}\bigr), & \text{object mode},\\
p_i^W(q_t^{(n)}), & \text{ground mode}.
\end{cases}
$$

robot-side object local points 记为：

$$
O=\{o_1,
\dots,o_{N_o}\}.
$$

当前 robot-side 顶点矩阵为：

$$
V_t^{rob}(q_t^{(n)})=
\begin{bmatrix}
p_1^{X_t}(q_t^{(n)})^\top\\
\vdots\\
p_{N_r}^{X_t}(q_t^{(n)})^\top\\
o_1^\top\\
\vdots\\
o_{N_o}^\top
\end{bmatrix}
\in\mathbb{R}^{N\times 3}.
$$

当前 robot-side Laplacian 为：

$$
\ell_t^{(n)} = \operatorname{vec}\left(L_t V_t^{rob}(q_t^{(n)})\right)\in\mathbb{R}^{3N}.
$$

代码变量对应关系为：

- `robot_pts_local`：$p_i^{X_t}(q_t^{(n)})$；
- `obj_pts_local`：$o_j$；
- `vertices`：$V_t^{rob}(q_t^{(n)})$；
- `lap0_vec`：$\ell_t^{(n)}$。

### 3.3 Laplacian 线性化

`_calc_manipulator_jacobians(...)` 对每个 robot keypoint 给出 active slice 上的位置 Jacobian：

$$
J_{p_i}^{(n)} =
\frac{\partial p_i^{X_t}}{\partial q_a}(q_t^{(n)})
\in\mathbb{R}^{3\times n_a}.
$$

将 robot keypoint Jacobian 堆叠，并为 object local points 填零：

$$
J_V^{(n)}=
\begin{bmatrix}
J_{p_1}^{(n)}\\
\vdots\\
J_{p_{N_r}}^{(n)}\\
0_{3N_o\times n_a}
\end{bmatrix}
\in\mathbb{R}^{3N\times n_a}.
$$

代码中 `J_V` 初始化为 `3 * V` 行，只有前 `3 * V_r` 行被 robot keypoint Jacobian 填充，object point 行保持零。

定义：

$$
J_{L,t}^{(n)}=(L_t\otimes I_3)J_V^{(n)}\in\mathbb{R}^{3N\times n_a}.
$$

于是当前迭代点附近的 robot-side Laplacian 近似为：

$$
\operatorname{vec}\left(L_tV_t^{rob}(q_t^{(n)}+\Delta q_{a,t})\right)
\approx
\ell_t^{(n)} + J_{L,t}^{(n)}\Delta q_{a,t}.
$$

代码引入辅助变量：

$$
z_t\in\mathbb{R}^{3N},
$$

对应 `lap_var`，并用线性等式把它绑定到线性化 Laplacian：

$$
z_t = \ell_t^{(n)} + J_{L,t}^{(n)}\Delta q_{a,t}.
$$

代码中的约束写作：

```python
J_L @ dqa - lap_var == -lap0_vec
```

等价于上式。

实现注记：当前代码在 `J_XC_dict` 中已经把 keypoint Jacobian 切到 `self.q_a_indices`，后面又写了 `J_L[:, self.q_a_indices]`。默认 `q_a_init_idx=-7` 时 `self.q_a_indices` 从 0 开始，公式可按上面的 $J_{L,t}^{(n)}\in\mathbb{R}^{3N\times n_a}$ 理解。若把 `q_a_init_idx` 改成非默认值，需要单独检查这一处二次索引是否仍然符合预期。

### 3.4 Holosoma 中 Jacobian 的创建路径

Holosoma 这里的 Jacobian 不是有限差分，也不是自动微分生成的。代码先调用 MuJoCo 的解析运动学 Jacobian，得到点速度对 MuJoCo generalized velocity `qvel` 的导数，然后用手写的坐标变换矩阵把它转成对 `qpos` 坐标增量的导数，最后按 active slice 截取列并组装成 Laplacian、foot 和 collision 约束需要的矩阵。

#### 3.4.1 单个机器人点的位置 Jacobian

核心入口是 `_calc_manipulator_jacobians(q, links, obj_frame, point_offsets)`。它先把当前线性化点写入 MuJoCo：

```text
robot_data.qpos[:] = q
mujoco.mj_forward(robot_model, robot_data)
```

然后对 `links` 中的每个 keypoint：

1. 用 MuJoCo body name 找到 `body_id`；
2. 取 body-local point offset，默认是 body 原点 `pC_B = 0`；
3. 调用 `_calc_contact_jacobian_from_point(body_id, pC_B)` 得到该点的 full `qpos` Jacobian；
4. 如果需要 object frame 表达，再把 world-frame 位置和 Jacobian 左乘当前 object rotation 的转置；
5. 用 `self.q_a_indices` 只保留 active robot slice 的列。

对 body $B$ 上的局部点 $p_{C/B}^B$，代码先得到 world 点：

$$
p_C^W = p_B^W + R_{WB}p_{C/B}^B.
$$

随后 `_calc_contact_jacobian_from_point(...)` 调用：

```text
mujoco.mj_jac(model, data, Jp, Jr, p_C^W, body_id)
```

其中 `Jp` 是 MuJoCo 返回的平移 Jacobian：

$$
\dot p_C^W = J_{p,v}(q)v,
\qquad
J_{p,v}(q)\in\mathbb{R}^{3\times n_v}.
$$

`Jr` 也被分配出来传给 MuJoCo，但当前代码不使用旋转 Jacobian；后续只用 `Jp`。

#### 3.4.2 从 MuJoCo `qvel` Jacobian 转成 `qpos` Jacobian

局部优化变量 `dqa` 是 `qpos` active slice 的坐标增量，不是 MuJoCo `qvel`。MuJoCo 的 `qpos` 和 `qvel` 在 free joint / ball joint 存在时不是同一个坐标空间：free joint 的 `qpos` 用 7 维 $(p,r)$ 表示位姿，而 `qvel` 用 6 维 $(v_{lin},\omega)$ 表示速度。因此，不能直接把 `mj_jac` 返回的 $3\times n_v$ 矩阵拿来右乘 `dqa`。

代码先引入 `qpos` 坐标曲线 $q(s)$。这里 $s$ 只是用于定义导数的局部曲线参数，不是动作帧索引 $t$。在当前线性化点 $q=q_t^{(n)}$，令：

$$
\dot q = \frac{dq(s)}{ds}\bigg|_{s=0}.
$$

MuJoCo generalized velocity $v$ 与这个 `qpos` 坐标导数之间满足局部关系。当前代码用 `_build_transform_qdot_to_qvel_fast()` 构造：

$$
v = T(q)\dot q,
\qquad
T(q)\in\mathbb{R}^{n_v\times n_q}.
$$

现在对点位置 $p_C^W(q)$ 应用链式法则。MuJoCo `mj_jac` 给出的定义是：

$$
\frac{d}{ds}p_C^W(q(s)) = J_{p,v}(q)\,v.
$$

又因为当前代码选择用 `qpos` 坐标导数表达同一条局部曲线的速度，$v=T(q)\dot q$，所以：

$$
\frac{d}{ds}p_C^W(q(s))
=J_{p,v}(q)T(q)\dot q.
$$

这就是为什么把 MuJoCo 的 `qvel` Jacobian 右乘这个矩阵后，可以得到相对于 `qpos` 坐标导数的 Jacobian：

$$
J_{p,\dot q}(q)=J_{p,v}(q)T(q)
\in\mathbb{R}^{3\times n_q}.
$$

`T(q)` 的块结构由 MuJoCo joint type 决定：

- free root 的平移块是单位阵：$v_{lin}=\dot p$；
- free root 的四元数块是 $\omega_W=2E_W(r)\dot r$，默认使用 world-frame angular velocity；
- 如果存在动态物体 free joint，代码对第二个 free joint 也填同样的 $6\times 7$ 块；
- hinge 和 slide joint 使用 $v_i=\dot q_i$；
- ball joint 分支没有实现，遇到会抛出 `NotImplementedError`。

这里的 free-joint 四元数块使用当前线性化点的四元数 $r=(r_w,r_x,r_y,r_z)$：

$$
E_W(r)=
\begin{bmatrix}
-r_x & r_w & r_z & -r_y\\
-r_y & -r_z & r_w & r_x\\
-r_z & r_y & -r_x & r_w
\end{bmatrix},
\qquad
T_{\omega r}=2E_W(r).
$$

局部凸子问题使用的是有限小增量 $\Delta q_{a,t}$，而不是显式的连续导数 $\dot q$。一阶线性化把二者连接起来：取一条局部曲线 $q(s)=q+s\Delta q+O(s^2)$，则 $\dot q|_{s=0}=\Delta q$，于是：

$$
p_C^W(q+\Delta q)
\approx
p_C^W(q)+J_{p,\dot q}(q)\Delta q.
$$

再截取 active slice 后，实际进入优化问题的是：

$$
\delta p_C^W
\approx
J_{p,a}(q)\Delta q_{a,t},
\qquad
J_{p,a}(q)=J_{p,\dot q}(q)[:,\mathcal A].
$$

所以，单个点最终返回的 full Jacobian 是：

$$
J_C^W(q)=J_{p,v}(q)T(q).
$$

这一步说明代码在创建 Jacobian 时考虑了 MuJoCo free-joint 的四元数微分关系；但局部优化求出的仍是小的 `qpos` 坐标增量，后续更新还是 ambient 加法再归一化。

MuJoCo Python API 没有在 `mj_jac` 旁边提供一个直接返回 $\partial p/\partial qpos$ 的高层函数。MuJoCo 的标准解析 Jacobian 是按 generalized velocity `qvel` 定义的。若局部变量选 `qvel`，可以直接使用 `Jp`；若局部变量选 `qpos` 坐标增量，就需要像当前代码这样显式构造 `qpos` 坐标导数到 `qvel` 的映射，或者用有限差分近似。MuJoCo 另外提供 `mj_differentiatePos` / `mj_integratePos` 这类位置差分和积分工具，但它们给的是 `qpos` 与 `qvel` 之间的运动学操作，不是点位置对 `qpos` 的完整解析 Jacobian。

Drake 的接口更显式地区分 Jacobian 的变量类型。例如 MultibodyPlant 的 translational velocity Jacobian API 可以选择相对于 generalized velocity $v$ 或相对于 $\dot q$ 的 Jacobian，并且 Drake 还提供 `MapQDotToVelocity` / `MapVelocityToQDot`。因此在 Drake 中通常不需要像这里这样手写每个 MuJoCo joint type 的 $T(q)$ 块，但仍然必须说清楚当前 Jacobian 是相对于 $v$、$\dot q$ 还是有限 position increment。

#### 3.4.3 代码凭什么把 `Jp @ T` 当作这个 Jacobian

当前实现没有在 `solve_single_iteration(...)` 里用额外算法从数据中“识别”或“估计”这个 Jacobian。它把 `Jp @ T` 当作 Jacobian 的依据是三层定义相互对齐：

1. `mujoco.mj_jac(...)` 的 API 契约定义了 `Jp` 的含义。给定当前 `robot_data.qpos`、执行 `mj_forward` 后的 kinematic state、world point `p_W` 和 `body_id`，`Jp` 满足 $\dot p_C^W=J_{p,v}(q)v$。这一步确认的是 `Jp` 的列空间对应 MuJoCo `qvel`，不是 `qpos`。
2. `_build_transform_qdot_to_qvel_fast()` 按 MuJoCo joint type 和 joint address 填块，定义了当前代码使用的速度坐标转换 $v=T(q)\dot q$。free joint 的平移块是单位阵，四元数块是 $2E_W(r)$，hinge / slide joint 是单位标量块。这个函数确认的是 `T` 的行对应 `qvel`，列对应 `qpos` 坐标导数。
3. `_calc_contact_jacobian_from_point(...)` 只做矩阵复合：`return Jp @ T`。由于 `Jp` 把 `qvel` 映射到点速度，`T` 把 `qpos` 坐标导数映射到 `qvel`，右乘后的矩阵自然把 `qpos` 坐标导数映射到点速度。这就是链式法则，维度也唯一匹配：

$$
(3\times n_v)(n_v\times n_q)=3\times n_q.
$$

因此，代码层面真正确认的是每个中间量的语义和维度：

```text
Jp:  qvel -> point velocity
T:   qpos coordinate rate -> qvel
Jp @ T: qpos coordinate rate -> point velocity
```

随后一阶线性化把 coordinate rate 的 Jacobian 用到有限小增量上：

$$
p(q+\Delta q)-p(q)=J_{p,\dot q}(q)\Delta q+O(\|\Delta q\|^2).
$$

这不是逆运动学 Jacobian $\partial q/\partial p$，也不是通过求解器反推出来的矩阵；它是 forward kinematics 的方向导数 $\partial p/\partial q$ 在当前坐标参数化下的表示。

如果要检查这段实现是否和 MuJoCo 当前模型一致，可以做有限差分方向导数测试。取一个小的 active 增量 $\Delta q_a$，扩展成 full increment $\Delta q$，令：

$$
\Delta q[\mathcal A]=\Delta q_a,
\qquad
\Delta q[\bar{\mathcal A}]=0.
$$

再比较：

$$
\frac{p(q+\epsilon\Delta q)-p(q)}{\epsilon}
\quad\text{与}\quad
J_{p,a}(q)\Delta q_a.
$$

当 $\epsilon$ 足够小且四元数 perturbation / normalization 的处理与代码更新规则一致时，两者应在一阶误差范围内接近。这个有限差分测试是验证实现的手段；当前 retargeter 主流程没有每次求解都执行这个测试。

#### 3.4.4 object-frame keypoint Jacobian

interaction mesh 的 robot keypoint 在 object 模式下表达在 augmented object frame 中。设当前 object pose 是 $(p_O^W,R_{WO})$，则位置变换为：

$$
p_C^O(q)=R_{WO}^\top\left(p_C^W(q)-p_O^W\right).
$$

代码中的 Jacobian 变换是：

$$
J_C^O(q)=R_{WO}^\top J_C^W(q).
$$

这里没有对 $R_{WO}$ 或 $p_O^W$ 再求导，因为当前局部子问题里 object pose 是锁定的，`dqa` 也只覆盖机器人 active slice。换句话说，object frame 只是当前线性化点处的固定表达坐标系。

最终 `_calc_manipulator_jacobians(...)` 存入字典的是 active-slice Jacobian：

$$
J_{C,a}^X(q)=J_C^X(q)[:,\mathcal{A}]
\in\mathbb{R}^{3\times n_a},
$$

代码对应：

```text
J_XC_dict[name] = J_XC[:, self.q_a_indices]
p_XC_dict[name] = p_XC
```

foot sticking 和 foot-lock 也复用同一个函数，只是调用时 `obj_frame=False`，因此 foot Jacobian 保持 world-frame 表达。

#### 3.4.5 Laplacian Jacobian 的组装

`solve_single_iteration(...)` 先调用 `_calc_manipulator_jacobians(...)` 得到每个 robot keypoint 的 $3\times n_a$ Jacobian，再按 interaction mesh 顶点顺序堆叠：

$$
J_V=
\begin{bmatrix}
J_{C_1,a}^X\\
\vdots\\
J_{C_{N_r},a}^X\\
0_{3N_o\times n_a}
\end{bmatrix}.
$$

object local points 自身不随 `dqa` 改变，因此它们对应的顶点位置 Jacobian 行是零。注意这只表示 object 顶点自己的坐标不变；由于 Laplacian 行会和邻居顶点相减，object 顶点的 Laplacian 坐标仍可能通过机器人邻居的 Jacobian 依赖 `dqa`。

Laplacian 坐标满足：

$$
\operatorname{vec}(LV)=(L\otimes I_3)\operatorname{vec}(V),
$$

所以代码构造：

$$
J_L=(L\otimes I_3)J_V.
$$

这就是 Laplacian equality constraint 中使用的 Jacobian。它把 active configuration increment 映射成 interaction-mesh Laplacian 坐标的一阶变化：

$$
\delta\ell \approx J_L\Delta q_a.
$$

#### 3.4.6 collision distance Jacobian

非穿透和自碰撞约束使用的不是 $3\times n_a$ 点位置 Jacobian，而是 signed distance 的一维方向导数。创建路径是：

```text
_update_jacobians_and_phis_from_q(q)
  -> _prefilter_pairs_with_mj_collision(...)
  -> mujoco.mj_geomDistance(..., fromto)
  -> _compute_jacobian_for_contact_relative(...)
```

`mj_geomDistance` 把两个最近点写到 `fromto`：

$$
p_A^W=\texttt{fromto}[0:3],
\qquad
p_B^W=\texttt{fromto}[3:6].
$$

代码用最近点方向构造 signed-distance 法向：

$$
\hat n_{BA}^W=\operatorname{sign}(\phi)\frac{p_A^W-p_B^W}{\|p_A^W-p_B^W\|_2},
$$

退化情况下如果 pair 中有 ground，则使用预设的竖直 ground normal；否则使用零向量作为 fallback。

然后对两个最近点分别调用 `_calc_contact_jacobian_from_point(..., input_world=True)`。这里 `input_world=True` 表示传入点已经是 world 坐标，不再做 body-local 到 world 的转换。得到：

$$
J_A=\frac{\partial p_A^W}{\partial q},
\qquad
J_B=\frac{\partial p_B^W}{\partial q}.
$$

相对位移 Jacobian 是：

$$
J_{rel}=J_A-J_B.
$$

signed distance 的一阶 Jacobian 是把相对位移 Jacobian 投影到法向上：

$$
J_\phi=\hat n_{BA}^{W\top}J_{rel}\in\mathbb{R}^{1\times n_q}.
$$

加入优化问题时再截取 active slice：

$$
J_{\phi,a}=J_\phi[:,\mathcal{A}],
$$

代码里一维数组写作 `Ja_n_full[self.q_a_indices]`。object / ground non-penetration 和 self-collision 都复用这条 signed-distance Jacobian 路径；区别只在候选 geom pair 的来源和约束右端项。

#### 3.4.7 维度汇总

| 对象 | 代码来源 | 维度 | 含义 |
|---|---|---:|---|
| `Jp` | `mujoco.mj_jac` | $3\times n_v$ | 点平移速度对 MuJoCo `qvel` 的 Jacobian |
| `T` | `_build_transform_qdot_to_qvel_fast` | $n_v\times n_q$ | `qpos` 坐标导数到 MuJoCo `qvel` 的映射 |
| `J = Jp @ T` | `_calc_contact_jacobian_from_point` | $3\times n_q$ | 点位置对 `qpos` 坐标导数 / 一阶 `qpos` 增量的 Jacobian |
| `J_XC[:, self.q_a_indices]` | `_calc_manipulator_jacobians` | $3\times n_a$ | active slice 上实际右乘 `dqa` 的 keypoint Jacobian |
| `J_V` | `solve_single_iteration` | $3N\times n_a$ | interaction-mesh 顶点位置 Jacobian，object 点行补零 |
| `J_L` | `(L \otimes I_3) @ J_V` | $3N\times n_a$ | Laplacian 坐标 Jacobian |
| `J_phi` | `nhat @ (J_A - J_B)` | $1\times n_q$ | signed distance 对 full `qpos` 的 Jacobian |
| `Ja_n` | `J_phi[self.q_a_indices]` | $1\times n_a$ | signed distance 对 active slice 的 Jacobian |

因此，Holosoma 中所谓创建 Jacobian 的实际流程可以概括为：

```text
MuJoCo analytic point Jacobian wrt qvel
-> right-multiply T(q) to get the first-order qpos-coordinate Jacobian
-> optionally rotate into object frame
-> slice active robot columns
-> stack or project depending on objective / constraint type
```

### 3.5 完整局部凸子问题

固定帧 $t$、固定内层迭代点 $q_t^{(n)}$ 时，当前代码实际构造的局部问题可以写为：

$$
\begin{aligned}
\min_{\Delta q_{a,t},\,z_t}\quad
& \left\|W_L^{1/2}(z_t-\hat\ell_t)\right\|_2^2 \\
& + \mathbf{1}_{nom}\,w_{nom,t}\left\|\Pi\left(\Delta q_{a,t}-(q_{a,t}^{nom}-q_{a,t}^{(n)})\right)\right\|_2^2 \\
& + \left\|Q^{1/2}\left(\Delta q_{a,t}+q_{a,t}^{(n)}\right)\right\|_2^2 \\
& + \rho_s\left\|\Delta q_{a,t}-\left(q_{a,t-1}-q_{a,t}^{(n)}\right)\right\|_2^2
\end{aligned}
$$

subject to

$$
z_t = \ell_t^{(n)} + J_{L,t}^{(n)}\Delta q_{a,t},
$$

$$
b_{F,k}^{xy,-}\le J_{F,k,xy}^{(n)}\Delta q_{a,t}\le b_{F,k}^{xy,+}
\qquad \text{for active foot-sticking points},
$$

$$
b_{F,k}^{z,-}\le J_{F,k,z}^{(n)}\Delta q_{a,t}\le b_{F,k}^{z,+}
\qquad \text{for active foot-lock windows},
$$

$$
J_{c}^{(n)}\Delta q_{a,t}\ge -\phi_c^{(n)}-\varepsilon_{pen}
\qquad \text{for active object/ground collision pairs},
$$

$$
J_{sc}^{(n)}\Delta q_{a,t}\ge \tau_{sc}-\phi_{sc}^{(n)}
\qquad \text{for active self-collision pairs},
$$

$$
q_{a,lb}-q_{a,t}^{(n)}\le \Delta q_{a,t}\le q_{a,ub}-q_{a,t}^{(n)},
$$

$$
\left\|\Delta q_{a,t}\right\|_2\le \varepsilon.
$$

其中：

- $W_L=\operatorname{diag}(w_v\otimes \mathbf{1}_3)$，代码里 `w_v = self.laplacian_weights * np.ones(V)`，默认 `self.laplacian_weights = 10`。
- $\mathbf{1}_{nom}=1$ 当且仅当 `w_nominal_tracking > 0` 且 `q_a_nominal is not None` 且 nominal index set 非空；否则该项不加入问题。
- $w_{nom,t}$ 对应 `w_nominal_tracking`。如果 `original=True`，取 `w_nominal_tracking_init`；否则取 `w_nominal_tracking_init * exp(-t / nominal_tracking_tau)`。
- $Q=\operatorname{diag}(Q_{diag})$。默认 `Q_diag` 先被置零，再由 `task_constants.MANUAL_COST` 指定少量非零关节权重。
- $\rho_s$ 对应 scalar `self.smooth_weight`，默认值为 0.2。若 `smooth_weight` 被设置为向量或矩阵，代码改用对应的逐元素加权平方或 `quad_form`。
- $\varepsilon$ 对应 `self.step_size`，默认值为 0.2。
- $\varepsilon_{pen}$ 对应 `self.penetration_tolerance`。
- $\tau_{sc}$ 对应 `self._self_collision_tolerance`。

该问题是凸的：目标由二次平方项组成，约束为线性等式、线性不等式、box 约束和一个二阶锥 trust region。代码用 `cp.Problem(cp.Minimize(cp.sum(obj_terms)), constraints)` 构造，再调用 `problem.solve(solver=cp.CLARABEL, ...)`。

若第一帧 `init_t=True` 且 Clarabel 未返回 `OPTIMAL` 或 `OPTIMAL_INACCURATE`，代码会移除 `SOC(self.step_size, dqa)` 这个 trust-region 约束后重试一次。其他帧不做这个 fallback。


## 4. 目标项与约束解释

### 4.1 Interaction-mesh Laplacian matching

目标项：

$$
\left\|W_L^{1/2}(z_t-\hat\ell_t)\right\|_2^2.
$$

它要求 robot-side interaction mesh 的 Laplacian 坐标接近 source interaction mesh 的 Laplacian 坐标。

由于 $z_t$ 被约束为：

$$
z_t = \ell_t^{(n)} + J_{L,t}^{(n)}\Delta q_{a,t},
$$

该项等价于在线性化模型里最小化：

$$
\left\|W_L^{1/2}\left(\ell_t^{(n)}+J_{L,t}^{(n)}\Delta q_{a,t}-\hat\ell_t\right)\right\|_2^2.
$$

代码选择显式引入 `lap_var`，而不是直接把上式展开到 `dqa` 上。这样做使 Laplacian matching 的几何量和线性化等式分开表达：

- `lap_var` 表示当前局部模型预测出来的 robot-side Laplacian；
- `target_lap_vec` 表示 source-side Laplacian target；
- 线性等式表示 `lap_var` 必须来自当前迭代点的一阶几何模型。

这仍然是局部模型。原始非线性量是 $\operatorname{vec}(L_tV_t^{rob}(q))$，代码每次在 $q_t^{(n)}$ 处线性化它。

### 4.2 Nominal tracking

目标项：

$$
w_{nom,t}\left\|\Pi\left(\Delta q_{a,t}-(q_{a,t}^{nom}-q_{a,t}^{(n)})\right)\right\|_2^2.
$$

因为更新后的 active slice 是：

$$
q_{a,t}^{new}=q_{a,t}^{(n)}+\Delta q_{a,t},
$$

所以括号内可以改写为：

$$
\Delta q_{a,t}-(q_{a,t}^{nom}-q_{a,t}^{(n)})
=q_{a,t}^{new}-q_{a,t}^{nom}.
$$

因此 nominal tracking 实际惩罚的是：

$$
w_{nom,t}\left\|\Pi(q_{a,t}^{new}-q_{a,t}^{nom})\right\|_2^2.
$$

代码中的 $\Pi$ 由索引切片实现：

```python
idx = np.array(self.track_nominal_indices, dtype=int)
z = dqa[idx] - (q_a_nominal[idx] - q_a_n_last[idx])
obj_terms.append(w_nominal_tracking * cp.sum_squares(z))
```

该项只在 `q_nominal_list` 存在时生效。`robot_only` 普通运行通常传入 `q_nominal_list=None`，因此 nominal tracking 不加入优化。object interaction 或 climbing 的 augmentation 模式会从原始结果文件读取 `qpos` 作为 nominal trajectory；此时该项把 augmented 求解结果拉向原始 motion 对应的 robot trajectory。

该项是软约束，不是硬约束。优化可以为了 Laplacian matching、接触或非穿透约束偏离 nominal trajectory，只是会付出二次代价。

### 4.3 Manual `Q_diag` joint cost

目标项：

$$
\left\|Q^{1/2}\left(\Delta q_{a,t}+q_{a,t}^{(n)}\right)\right\|_2^2
=
\left\|Q^{1/2}q_{a,t}^{new}\right\|_2^2.
$$

这个项把被选中的 active slice 分量拉向 0，而不是拉向 nominal trajectory。代码中：

```python
self.Q_diag = np.zeros(self.nq_a) * 1e-3
self.Q_diag[np.array(list(self.task_constants.MANUAL_COST.keys())).astype(int)] = list(
    self.task_constants.MANUAL_COST.values()
)
```

`np.zeros(self.nq_a) * 1e-3` 仍然是全零向量。因此只有 `MANUAL_COST` 指定的索引有非零权重。默认 `g1` 配置中，manual cost 主要用于 waist yaw / waist roll 一类容易出现不自然姿态的自由度。

该项和 nominal tracking 的区别：

- nominal tracking 的参考是 $q_{a,t}^{nom}$，并且可随帧变化；
- `Q_diag` 的参考是 0，且由少量人工指定的权重控制；
- nominal tracking 可通过 `q_nominal_list is None` 完全关闭；
- `Q_diag` 只要 `MANUAL_COST` 非空就始终加入目标。

### 4.4 Temporal smoothing

目标项：

$$
\rho_s\left\|\Delta q_{a,t}-\left(q_{a,t-1}-q_{a,t}^{(n)}\right)\right\|_2^2.
$$

因为 $q_{a,t}^{new}=q_{a,t}^{(n)}+\Delta q_{a,t}$，所以：

$$
\Delta q_{a,t}-\left(q_{a,t-1}-q_{a,t}^{(n)}\right)
=q_{a,t}^{new}-q_{a,t-1}.
$$

该项等价于惩罚当前帧更新后的姿态偏离上一帧最终姿态：

$$
\rho_s\left\|q_{a,t}^{new}-q_{a,t-1}\right\|_2^2.
$$

代码变量：

```python
dqa_smooth = q_t_last[self.q_a_indices] - q_a_n_last
obj_terms.append(self.smooth_weight * cp.sum_squares(dqa - dqa_smooth))
```

这里的 `q_t_last` 不是当前帧上一轮内层迭代，而是上一帧已接受的结果。该项提供 frame-to-frame 平滑，同时也为欠定的姿态自由度提供一个近邻锚点。

### 4.5 Laplacian equality constraint

约束：

$$
z_t = \ell_t^{(n)} + J_{L,t}^{(n)}\Delta q_{a,t}.
$$

它把辅助变量 $z_t$ 绑定为 robot-side Laplacian 的一阶近似。没有这个约束时，$z_t$ 会直接等于 $\hat\ell_t$ 使 Laplacian objective 为零，而不对机器人姿态产生任何作用。

### 4.6 Foot sticking 约束

当 `activate_foot_sticking=True` 且 `q_a_init_idx < 12` 时，代码根据 `foot_sticking_sequences[t]` 决定哪些脚处于 sticking 状态。

对处于 sticking 状态的 foot link $k$，记当前迭代点处脚点 world 位置为：

$$
p_{F,k}^{(n)}\in\mathbb{R}^3,
$$

上一帧最终位置为：

$$
p_{F,k}^{prev}\in\mathbb{R}^3.
$$

脚点位置 Jacobian 的 XY 块为：

$$
J_{F,k,xy}^{(n)}\in\mathbb{R}^{2\times n_a}.
$$

代码构造：

$$
p_{F,k}^{prev}-p_{F,k}^{(n)}-\tau_f\mathbf{1}
\le
J_{F,k}^{(n)}\Delta q_{a,t}
\le
p_{F,k}^{prev}-p_{F,k}^{(n)}+\tau_f\mathbf{1},
$$

随后只取 XY 两维：

$$
\left(p_{F,k}^{prev}-p_{F,k}^{(n)}-\tau_f\mathbf{1}\right)_{xy}
\le
J_{F,k,xy}^{(n)}\Delta q_{a,t}
\le
\left(p_{F,k}^{prev}-p_{F,k}^{(n)}+\tau_f\mathbf{1}\right)_{xy}.
$$

其中 $\tau_f$ 对应 `foot_sticking_tolerance`。该约束使当前帧 sticking foot 的 XY 位置保持在上一帧位置附近。它不是直接跟踪人体脚的位置，而是锁住机器人自己的上一帧脚位置。

### 4.7 Foot-lock Z 约束

当 `foot_lock.enable=True` 且当前 frame index 落入配置的 foot-lock window 时，代码对相应脚点的 Z 坐标加入 floor pinning：

$$
z_{floor}-p_{F,k,z}^{(n)}-\tau_{lock}
\le
J_{F,k,z}^{(n)}\Delta q_{a,t}
\le
z_{floor}-p_{F,k,z}^{(n)}+\tau_{lock}.
$$

其中：

- $z_{floor}$ 对应 `foot_lock.z_floor`；
- $\tau_{lock}$ 对应 `foot_lock.tolerance`。

这与 foot sticking 的 XY 约束互补：foot sticking 主要固定水平位置，foot-lock window 可在指定帧段内把脚点 Z 坐标钉到地面高度附近。

### 4.8 Object / ground non-penetration 约束

`_update_jacobians_and_phis_from_q(q)` 先把当前 $q_t^{(n)}$ 写入 MuJoCo，再通过 `mj_collision` 和 `mj_geomDistance` 找到距离小于阈值的候选 geom pair。

对一个候选 collision pair $c$，代码得到：

- signed distance：$\phi_c^{(n)}$；
- 相对距离方向上的 Jacobian：$J_c^{(n)}\in\mathbb{R}^{1\times n_a}$。

约束写为：

$$
J_c^{(n)}\Delta q_{a,t}\ge -\phi_c^{(n)}-\varepsilon_{pen}.
$$

等价于线性化 signed distance 满足：

$$
\phi_c^{(n)}+J_c^{(n)}\Delta q_{a,t}\ge -\varepsilon_{pen}.
$$

因此代码允许最多 $\varepsilon_{pen}$ 的线性化穿透余量，而不是强制距离非负。候选 pair 的过滤逻辑只保留与 object 或 ground 相关的 pair，并跳过 object-ground pair。

### 4.9 Self-collision 约束

如果 `SelfCollisionConfig.enable=True` 且配置了 body pair，代码会把 body pair 展开成 MuJoCo geom pair。对每个当前距离不大于检测阈值的 self-collision pair，得到：

- self distance：$\phi_{sc}^{(n)}$；
- 线性化 Jacobian：$J_{sc}^{(n)}$。

约束写为：

$$
J_{sc}^{(n)}\Delta q_{a,t}\ge \tau_{sc}-\phi_{sc}^{(n)},
$$

等价于：

$$
\phi_{sc}^{(n)}+J_{sc}^{(n)}\Delta q_{a,t}\ge \tau_{sc}.
$$

其中 $\tau_{sc}$ 是 self-collision 最小距离阈值。该约束只对启用的 pair、生效的 frame window、以及距离进入检测阈值的 geom pair 加入。

### 4.10 Joint limit 约束

如果 `activate_joint_limits=True`，代码加入：

$$
q_{a,lb}-q_{a,t}^{(n)}\le \Delta q_{a,t}\le q_{a,ub}-q_{a,t}^{(n)}.
$$

`q_a_lb` 和 `q_a_ub` 来自：

1. free root 的大范围边界 `[-1e6, 1e6]`；
2. MuJoCo actuated joint ranges；
3. `task_constants.MANUAL_LB` 和 `task_constants.MANUAL_UB` 对指定索引的覆盖。

默认配置还给根四元数 4 维加入人工边界：

$$
-1\le r_w,r_x,r_y,r_z\le 1.
$$

该边界不是单位四元数约束。真正使根四元数回到单位长度的是求解后的 normalization。

### 4.11 Trust region 约束

代码总是加入二阶锥约束：

$$
\left\|\Delta q_{a,t}\right\|_2\le \varepsilon.
$$

对应：

```python
constraints += [cp.SOC(self.step_size, dqa)]
```

它限制单次局部线性化的步长，防止线性模型在过大的 configuration increment 上失效。第一帧如果因为 trust region 导致求解失败，代码会移除该约束重试。


## 5. 四元数更新

四元数处理需要分成三个层面：变量层、Jacobian 层、更新层。当前代码在这三层的处理方式不同。

### 5.1 变量层：四维 quaternion coordinate increment

默认 `q_a_init_idx=-7` 时，`dqa` 包含根四元数的 4 个 `qpos` 坐标增量：

$$
\Delta r =
\begin{bmatrix}
\Delta r_w & \Delta r_x & \Delta r_y & \Delta r_z
\end{bmatrix}^\top.
$$

局部问题没有显式加入：

$$
r^\top\Delta r = 0,
$$

也没有显式加入：

$$
\|r+\Delta r\|_2=1.
$$

因此从优化变量看，根四元数块是 ambient 4D coordinate increment，不是 3D tangent-vector increment。

### 5.2 Jacobian 层：MuJoCo qvel 到 qdot 的转换

MuJoCo 的 `mj_jac` 返回的是对 generalized velocity $v$ 的点速度 Jacobian。代码需要得到对 `qpos` coordinate rate $\dot q$，进而对局部小增量 $\Delta q$ 的一阶 Jacobian，于是构造矩阵：

$$
v = T(q)\dot q.
$$

`_build_transform_qdot_to_qvel_fast(...)` 中对 free root 的平移块使用：

$$
v_{lin}=\dot p.
$$

对 free root 的角速度块使用：

$$
\omega_W = 2E_W(r)\dot r,
$$

其中 $r=(r_w,r_x,r_y,r_z)$，代码中的 world-frame $E_W(r)$ 为：

$$
E_W(r)=
\begin{bmatrix}
-r_x & r_w & r_z & -r_y\\
-r_y & -r_z & r_w & r_x\\
-r_z & r_y & -r_x & r_w
\end{bmatrix}.
$$

代码实际填入的是 $2E_W(r)$：

$$
T_{\omega r}=2E_W(r).
$$

若使用 body-frame 调试分支，代码可改用另一个 $E_B(r)$，但默认 `use_world_omega=True`。

点 Jacobian 的计算关系为：

$$
J_{p,qdot}(q)=J_{p,v}(q)T(q),
$$

其中：

- $J_{p,v}$ 来自 MuJoCo `mj_jac`；
- $T(q)$ 来自 `_build_transform_qdot_to_qvel_fast()`；
- `Jp @ T` 返回对 `qpos` coordinate rate 的 translational Jacobian；在一阶线性化中同一个矩阵右乘小的 `qpos` 坐标增量。

这说明代码在微分层面没有完全忽略 free-joint quaternion 运动学。它用 $\omega_W=2E_W(r)\dot r$ 把四元数坐标导数映射到 MuJoCo generalized velocity 的角速度块。

### 5.3 更新层：加法更新后归一化

局部问题求解后，代码执行：

```python
q_star = np.copy(q)
q_star[self.q_a_indices] = dqa_star + q_a_n_last
q_star[3:7] /= np.linalg.norm(q_star[3:7]) + 1e-12
```

数学上是：

$$
q_{a,t}^{+}=q_{a,t}^{(n)}+\Delta q_{a,t}^{\star},
$$

然后对根四元数：

$$
r_t^{+}\leftarrow
\frac{r_t^{+}}{\|r_t^{+}\|_2+10^{-12}}.
$$

因此当前 `src/interaction_mesh_retargeter.py` 的 quaternion 策略是：

$$
\text{Jacobian 层使用 MuJoCo free-joint differential relation；更新层使用 ambient 加法，再投影回单位四元数球面。}
$$

它不是 E-TaSP 更新，不是指数映射，也不是显式 tangent-space retraction。E-TaSP 相关变体在 `src/interaction_mesh_retargeter_etasp.py` 等文件中另行实现。

### 5.4 与 E-TaSP 更新的关键差异

E-TaSP 风格更新通常把四元数增量分解到当前四元数的切空间。例如 `Retarget/retargeting_omniretarget_qp_etasp.py` 中，更新逻辑是：

$$
\eta = \Delta r - (r^\top\Delta r)r,
$$

并在必要时把 $\eta$ clip 到 $\|\eta\|\le 1$，然后构造：

$$
r^+ = \sqrt{1-\|\eta\|^2}\,r + \eta.
$$

这类更新把可行四元数结构写进 update rule。`src/interaction_mesh_retargeter.py` 没有这样做；它只在最后执行 normalization。

这个差异属于更新规则差异，不等同于 underlying retargeting objective 完全相同。比较两个求解器时，应分别说明：

- 局部子问题的变量和目标是否相同；
- 四元数增量如何解释；
- 求解后如何从局部增量生成下一个 configuration。


## 6. 和 `Retarget/` 当前 solver 的关系

### 6.1 不能把两边都简称为同一个 baseline

`holosoma_retargeting/` 是 upstream Holosoma 相关内容的参考副本。`Retarget/` 是当前主动实验工作区，用于 OmniRetarget 论文问题、局部 solver 变体、E-TaSP / KKT 思路和评估脚本。

因此 `InteractionMeshRetargeter` 不能直接称为 `Retarget/` 的 baseline problem。更准确的分层是：

1. OmniRetarget 论文中的 frame-wise nonlinear optimization model；
2. 论文中的 sequentially linearized SOCP subproblem；
3. `Retarget/` 当前实现的 robot-only / flat-ground / keypoint-Laplacian task layer；
4. `Retarget/` 中每个 solver 实际解的 QP 或 NLP 子问题；
5. `holosoma_retargeting/src/interaction_mesh_retargeter.py` 中的 Holosoma reference local convex subproblem。

本文说明的是第 5 层。它与第 1 层、第 2 层、第 3 层有概念联系，但不是同一个数学问题。

### 6.2 与 `Retarget/retargeting_omniretarget_qp.py`

`Retarget/retargeting_omniretarget_qp.py` 使用 Drake，局部变量是 generalized velocity：

$$
dq_v\in\mathbb{R}^{n_v}.
$$

代码通过 Drake 的：

$$
\dot q = M(q)dq_v,
$$

并以时间步长 $dt$ 更新：

$$
q^{new}=q^{bar}+M(q^{bar})dq_v\,dt.
$$

其 Laplacian 局部模型大致是：

$$
L P(q^{bar}) + L J_v(q^{bar})dq_v\,dt - \hat\ell_t,
$$

并构造 Gauss-Newton 风格的二次目标：

$$
w_L\left\|A_{lap}dq_v+b_{lap}\right\|_2^2.
$$

它还加入 regularization：

$$
w_R\left\|q^{bar}+M(q^{bar})dq_v\,dt-q_{prev}\right\|_2^2.
$$

与 `InteractionMeshRetargeter` 的主要差异：

- `InteractionMeshRetargeter` 的变量是 configuration increment $\Delta q_a$；`retargeting_omniretarget_qp.py` 的变量是 generalized velocity $dq_v$。
- `InteractionMeshRetargeter` 的 interaction mesh 可包含 object / terrain sample points；`retargeting_omniretarget_qp.py` 当前主要是 robot-only keypoint Laplacian。
- `InteractionMeshRetargeter` 有 `lap_var` 辅助变量；`retargeting_omniretarget_qp.py` 直接把线性化 residual 展开成 Hessian 和 linear term。
- `InteractionMeshRetargeter` 有 MuJoCo collision-based object/ground non-penetration、自碰撞、foot-lock window、trust region；`retargeting_omniretarget_qp.py` 当前主要有 stance、ground、position/velocity limit 约束。
- `InteractionMeshRetargeter` 求解后对 MuJoCo `qpos` 加法更新；`retargeting_omniretarget_qp.py` 用 Drake 的 `MapVelocityToQDot` 更新。

### 6.3 与 `Retarget/retargeting_omniretarget_qp_etasp.py`

`Retarget/retargeting_omniretarget_qp_etasp.py` 的局部变量是 generalized position increment：

$$
\Delta q\in\mathbb{R}^{n_q}.
$$

它的目标不是对 Laplacian residual 做完整 Gauss-Newton 二次近似，而是使用 first-order Laplacian linear term 加 proximal quadratic：

$$
\min_{\Delta q}\quad
\frac{1}{2}\Delta q^\top H_{prox}\Delta q
+
\left[
\frac{1}{\tau}(q^{bar}-q_{prev})
+2w_LJ_{lap}^\top(LP(q^{bar})-\hat\ell_t)
\right]^\top\Delta q.
$$

其中 $H_{prox}=\frac{1}{\tau}I$。这和 `InteractionMeshRetargeter` 的 Laplacian squared local residual 不同：

- `InteractionMeshRetargeter` 保留了线性化 Laplacian residual 的平方：

  $$
  \left\|W_L^{1/2}(\ell^{(n)}+J_L\Delta q_a-\hat\ell)\right\|_2^2.
  $$

- `retargeting_omniretarget_qp_etasp.py` 在该文件当前实现中只把 Laplacian 对目标的一阶梯度放进 linear term，二次项主要来自 proximal anchor。

它与 `InteractionMeshRetargeter` 在四元数更新上也不同。`retargeting_omniretarget_qp_etasp.py` 使用 E-TaSP 更新：

$$
\eta = \Delta r-(r^\top\Delta r)r,
\qquad
r^+=\sqrt{1-\|\eta\|^2}r+\eta,
$$

而 `InteractionMeshRetargeter` 使用 ambient 4D 加法后 normalization。

### 6.4 与 `Retarget/retargeting_omniretarget_qp_etasp_gauss_newton.py`

`Retarget/retargeting_omniretarget_qp_etasp_gauss_newton.py` 保留 E-TaSP quaternion update，但把 Laplacian 项改成 Gauss-Newton 风格二次近似：

$$
H \leftarrow H + 2w_LJ_{lap}^\top J_{lap},
\qquad
f \leftarrow f + 2w_LJ_{lap}^\top(LP(q^{bar})-\hat\ell_t).
$$

因此它在 Laplacian 目标近似上更接近 `InteractionMeshRetargeter` 的 squared linear residual，但仍有差异：

- 变量空间不同：完整 Drake position increment $\Delta q$ vs MuJoCo active slice increment $\Delta q_a$。
- interaction graph 不同：`Retarget/` 当前 solver 使用 BVH keypoint topology；Holosoma reference 可以加入 object points 并逐帧构造 source interaction mesh。
- 约束集合不同：`Retarget/` 当前 solver 是 stance / ground / position limit / velocity limit；Holosoma reference 还有 object/ground collision pair 线性化、自碰撞、foot-lock window、trust region。
- 更新规则不同：E-TaSP retraction vs additive plus normalization。

### 6.5 与 `Retarget/retargeting_omniretarget_nlp_baseline.py`

`Retarget/retargeting_omniretarget_nlp_baseline.py` 是直接 nonlinear baseline。它每帧直接以完整 $q$ 为决策变量，目标为：

$$
w_L\left\|LP(q)-\hat\ell_t\right\|_2^2
+w_R\left\|q-q_{prev}\right\|_2^2.
$$

它加入 hard unit-quaternion constraint：

$$
r^\top r=1,
$$

并用非线性的 stance 和 ground constraints。求解器为 SNOPT 或 IPOPT。

这与 `InteractionMeshRetargeter` 的关系是：

- `retargeting_omniretarget_nlp_baseline.py` 是直接非线性问题；
- `InteractionMeshRetargeter` 是逐帧、内层多次局部凸化；
- `retargeting_omniretarget_nlp_baseline.py` 的 unit quaternion 是硬约束；
- `InteractionMeshRetargeter` 的 unit quaternion 由求解后 normalization 保证。

### 6.6 对比表

| 维度 | `InteractionMeshRetargeter` | `Retarget/retargeting_omniretarget_qp.py` | `Retarget/retargeting_omniretarget_qp_etasp.py` | `Retarget/retargeting_omniretarget_qp_etasp_gauss_newton.py` | `Retarget/retargeting_omniretarget_nlp_baseline.py` |
|---|---|---|---|---|---|
| 代码栈 | MuJoCo + CVXPY + Clarabel | Drake + MathematicalProgram + OSQP | Drake + MathematicalProgram + OSQP | Drake + MathematicalProgram + OSQP | Drake + SNOPT/IPOPT |
| 每帧结构 | 内层多次局部凸化 | 内层多次 QP | 内层多次 QP | 内层多次 QP | 单个 nonlinear program |
| 决策变量 | active `qpos` 增量 $\Delta q_a$ + $z$ | generalized velocity $dq_v$ | position increment $\Delta q$ | position increment $\Delta q$ | full position $q$ |
| Laplacian 处理 | 辅助变量 + 线性化 residual 平方 | velocity 线性化 residual 平方 | Laplacian first-order gradient + proximal | Gauss-Newton residual 平方 + proximal | exact nonlinear residual |
| object / terrain interaction mesh | 支持 human + object/terrain points | 当前 Retarget task layer 不含 Holosoma object points | 同左 | 同左 | 同左 |
| nominal tracking | 可选，依赖 `q_nominal_list` | 无同名项 | 无同名项 | 无同名项 | 无同名项 |
| manual joint cost | `Q_diag` | 无同名项 | 无同名项 | 无同名项 | 无同名项 |
| temporal anchor | smooth 到上一帧 $q_{t-1}$ | regularization 到上一帧 | proximal 到上一帧 | proximal 到上一帧 | regularization 到上一帧 |
| foot constraint | foot sticking XY + optional foot-lock Z | stance constraint | stance constraint | stance constraint | nonlinear stance constraint |
| collision / ground | MuJoCo object/ground pair non-penetration | ground contact spheres | ground contact spheres | ground contact spheres | nonlinear ground constraint |
| self-collision | 可选 | 当前无对应项 | 当前无对应项 | 当前无对应项 | 当前无对应项 |
| trust region | $\|\Delta q_a\|_2\le\varepsilon$ | 当前无同型 SOC trust region | 当前无同型 SOC trust region | 当前无同型 SOC trust region | 无局部 trust region |
| quaternion | 加法更新后 normalization | Drake velocity update后 normalization | E-TaSP update | E-TaSP update | hard unit constraint |

### 6.7 使用这些实现做比较时的措辞

更严谨的表述：

- `InteractionMeshRetargeter` 是 Holosoma reference implementation 的局部凸化求解器，包含 interaction mesh、工程约束和 additive-normalized quaternion update。
- `Retarget/retargeting_omniretarget_qp.py` 是当前 `Retarget/` workspace 中针对 OmniRetarget task layer 的 generalized-velocity QP update 实现。
- `Retarget/retargeting_omniretarget_qp_etasp.py` 是同一 task layer 上的 position-increment / E-TaSP update 变体，但 Laplacian 项当前是一阶 linear term 加 proximal。
- `Retarget/retargeting_omniretarget_qp_etasp_gauss_newton.py` 是同一 task layer 上的 E-TaSP + Gauss-Newton Laplacian 二次近似变体。
- `Retarget/retargeting_omniretarget_nlp_baseline.py` 是当前 task layer 的直接 nonlinear baseline。

不严谨的表述：

- 把 `InteractionMeshRetargeter` 直接称为 OmniRetarget paper baseline；
- 把 `Retarget/` 的 E-TaSP 变体说成“只改了四元数更新”，而不说明 Laplacian 局部模型和变量空间是否也变了；
- 把 `holosoma_retargeting` 的完整工程约束和 `Retarget/` 的 flat-ground robot-only task layer 放在一起比较 solve time，而不说明 underlying local subproblem 不同。


## 7. 一帧内的算法摘要

固定帧 $t$ 后，当前实现执行如下局部迭代：

1. 从人体关键点和 object / ground points 构造 source interaction mesh，得到 $L_t$ 和 $\hat\ell_t$。
2. 以上一帧结果作为 warm start，得到当前内层初值 $q_t^{(0)}$。
3. 对 $n=0,1,\dots$：
   
   $$
   q_t^{(n)} \longrightarrow
   \left(\ell_t^{(n)},J_{L,t}^{(n)},J_F^{(n)},J_c^{(n)},J_{sc}^{(n)}\right)
   \longrightarrow
   \text{local convex problem}
   \longrightarrow
   \Delta q_{a,t}^{\star}.
   $$

4. 用 additive update 得到：

   $$
   q_{a,t}^{(n+1)}=q_{a,t}^{(n)}+\Delta q_{a,t}^{\star}.
   $$

5. 对根四元数做：

   $$
   r_t^{(n+1)}\leftarrow\frac{r_t^{(n+1)}}{\|r_t^{(n+1)}\|_2+10^{-12}}.
   $$

6. 如果当前 local problem 的 cost 与上一轮 cost 足够接近，`iterate(...)` 停止；否则继续下一轮。
7. 当前帧最终 $q_t$ 存入 `retargeted_motions`，作为下一帧的 `q_t_last`。

因此，一帧内的核心数学对象不是单个 IK residual，而是一个由 interaction-mesh Laplacian、nominal tracking、manual joint cost、temporal smoothing 和多类线性化硬约束共同组成的局部凸子问题。
